# 10 - Export all evaluated models

Export RF, XGBoost, CNN and CNN-BiLSTM with a catalog ranked by Test Macro-F1 from notebook 9. Hyperparameters were selected on Validation; Test ranking is a benchmark comparison, not an independent estimate after model selection.


## Bundle contract and provenance

Run notebook 9 with artifact snapshots before export. The bundle contains the final checkpoint, its fitted transforms, feature schema, preprocessing/configuration snapshots and evaluation provenance. No `selection.json`, legacy imputer, retraining or additional Test inference is used.


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import shutil
import tempfile
import math

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'benchmark_config.json').is_file())


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def export_bundle(root=ROOT, model=None, destination=None):
    """Export one evaluated model, defaulting to the benchmark winner."""
    root = Path(root).resolve()
    data, artifacts = root/'dataset/processed', root/'model/artifacts'
    report = root/'model/reports/final'
    evaluation = read_json(report/'evaluation.json')
    config = read_json(root/'benchmark_config.json')
    prepared = read_json(data/'prepared.json')
    classes = ['N', 'S', 'V', 'F']
    if evaluation.get('classes') != classes or config.get('classes') != classes:
        raise ValueError('Export requires the four-class N/S/V/F benchmark')
    if (evaluation.get('ranking_metric') != 'test_macro_f1'
            or evaluation.get('hyperparameters_selected_on') != 'validation'
            or evaluation.get('protocol') != config['protocol']):
        raise ValueError('Evaluation protocol does not match notebook 9')
    if evaluation.get('prepared_hash') != sha256(data/'prepared.json'):
        raise ValueError('Prepared dataset changed after evaluation; rerun notebook 9')
    config_hash = sha256(root/'benchmark_config.json')
    if evaluation.get('config_hash') != config_hash or prepared.get('config_hash') != config_hash:
        raise ValueError('Configuration changed or evaluation is old; rerun notebooks as needed')
    if evaluation.get('comparison_hash') != sha256(report/'comparison.csv'):
        raise ValueError('Comparison changed or lacks provenance; rerun notebook 9')
    winner = evaluation['benchmark_top_model']
    with (report/'comparison.csv').open(encoding='utf-8', newline='') as stream:
        comparison = list(csv.DictReader(stream))
    kinds = {'rf', 'xgb', 'cnn', 'cnn_bilstm'}
    if (len(comparison) != 4 or {r['model'] for r in comparison} != kinds
            or comparison[0]['model'] != winner or winner not in kinds):
        raise ValueError('Comparison and benchmark winner disagree')
    scores = [float(row['macro_f1']) for row in comparison]
    if any(not math.isfinite(value) or not 0 <= value <= 1 for value in scores):
        raise ValueError('Invalid Test Macro-F1 scores')
    if scores != sorted(scores, reverse=True):
        raise ValueError('Comparison must be ranked by descending Test Macro-F1')
    winner = winner if model is None else model
    if winner not in kinds:
        raise ValueError('Unknown model requested')
    rank = next(i+1 for i, row in enumerate(comparison) if row['model'] == winner)
    snapshots = evaluation.get('artifact_snapshots', {})
    if set(snapshots) != kinds:
        raise ValueError('Evaluation lacks checkpoint snapshots; rerun notebook 9')
    snapshot = snapshots[winner]
    source = (artifacts/snapshot['path']).resolve()
    if not source.is_relative_to(artifacts.resolve()) or source == artifacts.resolve():
        raise ValueError('Invalid checkpoint location')
    model_file = 'model.joblib' if winner == 'rf' else 'model.ubj' if winner == 'xgb' else 'model.pt'
    required = {model_file, 'info.json', 'transforms.joblib'}
    if set(snapshot['files']) != required:
        raise ValueError('Incomplete checkpoint snapshot')
    for name, checksum in snapshot['files'].items():
        if sha256(source/name) != checksum:
            raise ValueError(f'Evaluated artifact changed: {name}; rerun notebook 9')
    info = read_json(source/'info.json')
    if (info.get('model') != winner or info.get('classes') != classes
            or info.get('prepared_hash') != evaluation['prepared_hash']):
        raise ValueError('Final model metadata does not match evaluation')
    for name in ['preprocessing.json', 'feature_schema.json']:
        if sha256(data/name) != prepared['files'][name]:
            raise ValueError(f'Prepared artifact changed: {name}')
    state = read_json(data/'preprocessing.json')
    if state.get('config_hash') != config_hash or state.get('classes') != classes:
        raise ValueError('Preprocessing and configuration disagree')

    # Build completely before replacing a previous bundle; restore it on failure.
    artifacts.mkdir(parents=True, exist_ok=True)
    output = (artifacts/'exported' if destination is None else Path(destination)).resolve()
    if not output.is_relative_to(artifacts.resolve()) or output == artifacts.resolve():
        raise ValueError('Export destination must be inside model/artifacts')
    backup = output.with_name(output.name + '.previous')
    if backup.exists():
        raise FileExistsError('Review exported.previous before exporting again')
    staging = Path(tempfile.mkdtemp(prefix='export-staging-', dir=artifacts))
    try:
        for name in required:
            shutil.copy2(source/name, staging/name)
        for name in ['preprocessing.json', 'feature_schema.json']:
            shutil.copy2(data/name, staging/name)
        shutil.copy2(root/'benchmark_config.json', staging/'benchmark_config.json')
        shutil.copy2(report/'evaluation.json', staging/'evaluation.json')
        manifest = {'version': 2, 'model': winner, 'classes': classes,
            'protocol': config['protocol'], 'source_path': snapshot['path'],
            'rank': rank, 'test_macro_f1': scores[rank-1],
            'ranking_metric': evaluation['ranking_metric'],
            'hyperparameters_selected_on': 'validation', 'test_used_for_ranking': True,
            'files': {p.name: sha256(p) for p in staging.iterdir() if p.is_file()}}
        (staging/'bundle.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
        if output.exists():
            output.rename(backup)
        try:
            staging.rename(output)
        except Exception:
            if backup.exists():
                backup.rename(output)
            raise
        if backup.exists():
            shutil.rmtree(backup)
    finally:
        if staging.exists():
            shutil.rmtree(staging)
    return output, manifest


def export_all_models(root=ROOT):
    """Stage all four verified bundles; publish the catalog only as a complete set."""
    root = Path(root).resolve()
    artifacts = root/'model/artifacts'
    report = root/'model/reports/final'
    with (report/'comparison.csv').open(encoding='utf-8', newline='') as stream:
        comparison = list(csv.DictReader(stream))
    if len(comparison) != 4 or {r['model'] for r in comparison} != {'rf', 'xgb', 'cnn', 'cnn_bilstm'}:
        raise ValueError('Expected all four evaluated models')
    artifacts.mkdir(parents=True, exist_ok=True)
    output, backup = artifacts/'exported', artifacts/'exported.previous'
    if backup.exists():
        raise FileExistsError('Review exported.previous before exporting again')
    staging = Path(tempfile.mkdtemp(prefix='all-models-staging-', dir=artifacts))
    evaluation_hash = sha256(report/'evaluation.json')
    comparison_hash = sha256(report/'comparison.csv')
    entries = []
    try:
        for row in comparison:
            kind = row['model']
            folder, manifest = export_bundle(root, model=kind, destination=staging/kind)
            if manifest['files']['evaluation.json'] != evaluation_hash:
                raise ValueError('Evaluation changed during export')
            entries.append({'model': kind, 'path': kind, 'rank': manifest['rank'],
                            'test_macro_f1': manifest['test_macro_f1'],
                            'bundle_hash': sha256(folder/'bundle.json')})
        if sha256(report/'comparison.csv') != comparison_hash:
            raise ValueError('Comparison changed during export')
        catalog = {'version': 1, 'ranking_metric': 'test_macro_f1',
                   'evaluation_hash': evaluation_hash, 'comparison_hash': comparison_hash,
                   'models': entries}
        (staging/'catalog.json').write_text(json.dumps(catalog, indent=2), encoding='utf-8')
        if output.exists():
            output.rename(backup)
        try:
            staging.rename(output)
        except Exception:
            if backup.exists():
                backup.rename(output)
            raise
        if backup.exists():
            shutil.rmtree(backup)
    finally:
        if staging.exists():
            shutil.rmtree(staging)
    return output, catalog


## Export all four models

All bundles must pass provenance checks before replacing the previous exported directory.


In [ ]:
output, catalog = export_all_models()
print('Exported:', output)
for entry in catalog['models']:
    print(entry['rank'], entry['model'], 'Test Macro-F1:', entry['test_macro_f1'])


## Application

Run `python program/app.py`, select the exported folder and choose a model from the ranked dropdown. Individual version 2 bundle folders remain supported.
